# Chat Agent

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction LR
INIT --> CHAT
CHAT --> FINAL
```


### a) Create Agent

In [7]:
from gai.asm.agents import ChatAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper

from gai.lib.tests import make_local_tmp
import os
here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
from gai.messages import FileMonologue
monologue = FileMonologue(agent_name="ChatAgent",file_path=file_path)

# Create an artificial dialogue history for testing
from gai.messages import FileDialogue, MessagePydantic
messages = [
    MessagePydantic(**{
        'id': 'b1e5f98c-f6eb-47de-a6e2-387510d970f9', 
        'header': {
            'sender': 'User',
            'recipient': 'Sara',
            "timestamp": 1751308157.270983,
            "order": 0            
        }, 'body': {
            'type': 'chat.send',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0,
            'step_no': 0,
            'role': "user",
            'content': 'I love horror stories, are you familiar with them?',
        }
    }),
    MessagePydantic(**{
        'id': 'abbc7961-45dc-4973-aaf4-a6224ed35d37', 
        'header': {
            'sender': 'Sara',
            'recipient': 'User',
            "timestamp": 1751308167.3488164,
            "order": 1
        }, 'body': {
            'type': 'chat.reply',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0, 
            'step_no': 1,
            'chunk_no':10,
            'chunk':'<eom>',
            'role': "assistant",
            'content': 'Yes, I am familiar with horror stories. They are a fascinating genre that can evoke strong emotions and create a sense of suspense and fear. Do you have any specific horror stories in mind that you would like to discuss?'
        }
    })]
from gai.lib.constants import DEFAULT_GUID
file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(messages=messages,file_path=file_path)
recap = dialogue.extract_recap()

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ChatAgent(
    agent_name="ChatAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue,
    recap=recap
)

### reset monologue (optional)

In [8]:
monologue.reset()


### b) run

In [9]:
user_message="Tell me a one paragraph story."
resp=agent.run(user_message=user_message)
# Stream the response
async for chunk in resp:
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
            
# Update dialogue
dialogue.add_user_message(recipient="Sara", content=user_message)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)

Maya discovered the old music box tucked behind dusty books in her grandmother's attic, its delicate ballerina frozen mid-pirouette. When she wound the tiny key, expecting the usual tinkling melody, instead she heard her grandmother's young voice singing a lullaby in a language Maya didn't recognize. As the haunting tune filled the cramped space, photographs around the room began to shimmer and move like windows into the past—her grandmother as a little girl fleeing a war-torn country, clutching nothing but this very music box. Maya realized she wasn't just hearing a song, but a piece of her family's history that had been waiting decades to be discovered, and as the melody faded, she carefully closed the box, knowing she had found something far more precious than any inheritance.

MessagePydantic(id='bb0e163d-28a7-4ace-a15b-c92cecdf2295', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1752651950.4458463, order=1), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=0, step_no=1, message_id='00000000-0000-0000-0000-000000000000.45', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content="Maya discovered the old music box tucked behind dusty books in her grandmother's attic, its delicate ballerina frozen mid-pirouette. When she wound the tiny key, expecting the usual tinkling melody, instead she heard her grandmother's young voice singing a lullaby in a language Maya didn't recognize. As the haunting tune filled the cramped space, photographs around the room began to shimmer and move like windows into the past—her grandmother as a little girl fleeing a war-torn country, clutching nothing but this very music box. Maya realized she wasn't just hearing a song, but a pie

### c) continue

In [10]:
resp=agent.run()
# Stream the response
async for chunk in resp:
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
            
# Update dialogue
dialogue.add_user_message(recipient="Sara", content="Please continue.")
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)

I'm glad you enjoyed that story! I had fun weaving together the elements of family history, mystery, and that bittersweet feeling of discovering something meaningful from the past. 

Is there anything particular about the story that resonated with you? Or would you like me to explore a different type of story - perhaps something in a completely different genre or style? I'm curious what kind of narratives you find most engaging.

MessagePydantic(id='421edbb5-9863-4256-8145-1b5166a460b4', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1752651966.645386, order=3), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=1, step_no=1, message_id='00000000-0000-0000-0000-000000000000.67', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content="I'm glad you enjoyed that story! I had fun weaving together the elements of family history, mystery, and that bittersweet feeling of discovering something meaningful from the past. \n\nIs there anything particular about the story that resonated with you? Or would you like me to explore a different type of story - perhaps something in a completely different genre or style? I'm curious what kind of narratives you find most engaging."))

### c) Show monologue

In [11]:
import json

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = agent.fsm.monologue.get_total_size()
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "91dd8313-830c-472d-9458-51eb9e8a0ed9",
    "header": {
        "sender": "User",
        "recipient": "ChatAgent",
        "timestamp": 1752651957.7603045,
        "order": 2
    },
    "body": {
        "type": "monologue",
        "state_name": "CHAT",
        "step_no": 3,
        "content_type": "text",
        "role": "user",
        "content": "Please continue the conversation."
    }
}
{
    "id": "a1be5e70-ba2f-49f6-8f1a-89efa015b7db",
    "header": {
        "sender": "ChatAgent",
        "recipient": "User",
        "timestamp": 1752651963.0064147,
        "order": 3
    },
    "body": {
        "type": "monologue",
        "state_name": "CHAT",
        "step_no": 3,
        "content_type": "text",
        "role": "assistant",
        "content": [
            {
                "citations": null,
                "text": "I'm glad you enjoyed that story! I had fun weaving together the elements of 

### d) Show dialogue

In [12]:
for msg in dialogue.list_messages():
    print(f"{msg.header.sender}: {msg.body.content}")

User: Sara, Tell me a one paragraph story.
Sara: Maya discovered the old music box tucked behind dusty books in her grandmother's attic, its delicate ballerina frozen mid-pirouette. When she wound the tiny key, expecting the usual tinkling melody, instead she heard her grandmother's young voice singing a lullaby in a language Maya didn't recognize. As the haunting tune filled the cramped space, photographs around the room began to shimmer and move like windows into the past—her grandmother as a little girl fleeing a war-torn country, clutching nothing but this very music box. Maya realized she wasn't just hearing a song, but a piece of her family's history that had been waiting decades to be discovered, and as the melody faded, she carefully closed the box, knowing she had found something far more precious than any inheritance.
User: Sara, Please continue.
Sara: I'm glad you enjoyed that story! I had fun weaving together the elements of family history, mystery, and that bittersweet fee